In [ ]:
# scripts/run_param_sim.py
import argparse, os, time, json, glob
import numpy as np
import pandas as pd

from dina.config import EMConfig, QMipConfig, QSatConfig, Priors
from dina.em import DINAEM
from simulatedata import simulate_dina
from metrics import q_entry_accuracy, q_hamming, q_row_recovery, sg_rmse
from q_candidates import CANDIDATES



In [2]:
def _vec_like(val, J: int) -> np.ndarray:
    """Return a J-length vector; broadcast scalars."""
    if np.isscalar(val):
        return np.full(J, float(val), dtype=float)
    arr = np.asarray(val, dtype=float)
    if arr.shape == (J,):
        return arr
    if arr.size == 1:
        return np.full(J, float(arr.item()), dtype=float)
    raise ValueError(f"Cannot broadcast {arr.shape} to length {J}.")

In [3]:
Q_true = np.array([
    [1,0,0],
    [0,1,0],
    [0,0,1],
    [1,0,0],
    [0,1,0],
    [0,0,1],
    [1,0,0],
    [0,1,0],
    [0,0,1],
    [1,1,0],
    [1,0,1],
    [0,1,1],
    [1,1,0],
    [1,0,1],
    [0,1,1],
    [1,1,1],
    [1,1,1],
    [1,1,1],
], dtype=int)


J, K = Q_true.shape
rho = 0.25
N = 500
# s,g vectors
s_true = _vec_like(0.2, J)
g_true = _vec_like(0.2, J)


rng = np.random.default_rng(0)

R, A, xi = simulate_dina(
    N,
    Q_true,
    s_true,
    g_true,
    rho=rho,
    rng=rng,
)


In [4]:

# example bounds (unchanged logic; uses inferred J,K)
col_lb = [3] * K if K >= 3 else [1] * K
col_up = None
row_lb = [1] * J
row_up = None


In [ ]:

# -------------------------
# EM config
# -------------------------
em_cfg = EMConfig(
    multistart=20,
    max_iter=200,
    tol=1e-6,
    verbose=True,
)

em_cfg_2 = EMConfig(
    multistart=50,
    max_iter=200,
    tol=1e-6,
    verbose=True,
)


# -------------------------
# Q estimation (MIP) config
#   this controls your mstep_update_Q(...)
# -------------------------
q_cfg = QMipConfig(
    include_identity=True,          # if you want identity pinned/required
    include_distinctness=True,      # if you enforce distinct columns
    lexi=True,                      # symmetry breaking
    lexi_ascending=False,           # your convention (descending chain)
    lexi_strict=True,               # strict lex for estimation
    hierarchy_edges=None,
    time_limit=60.0,
    mip_gap=1e-4,
    log_to_console=False,
)

# -------------------------
# Q init sampling (SAT/MaxSAT) config
#   derived from q_cfg but relax strict lex for diversity
# -------------------------
q_sat_cfg = QSatConfig.from_qmip(
    q_cfg,
    frac_assumption_sat=0.75,        # 60–80%
    frac_sparse_maxsat=0.25,         # 20–40%
    frac_dense_maxsat=0.00,          # optional
    min_hamming_frac=0.05,           # enforce diversity among starts
    assumption_frac=0.25,
    tries_per_start=40,
    solver_name="g4",
)


em = DINAEM()

out = em.fit(
    R, K,
    em_cfg=em_cfg,
    q_cfg=q_cfg,
    q_sat_cfg=q_sat_cfg,    
    seed=2026,
)

out2 = em.fit(
    R, K,
    em_cfg=em_cfg_2,
    q_cfg=q_cfg,
    q_sat_cfg=q_sat_cfg,    
    seed=2026,
)

Q_hat = out["Q"]
s_hat = out["s"]
g_hat = out["g"]
ll_hat = out["loglik"]

print("loglik:", ll_hat)
print("Q_hat shape:", Q_hat.shape)
print("s_hat[:5]:", s_hat[:5])
print("g_hat[:5]:", g_hat[:5])


[EM] Q-init sampling failed (integer expected); falling back to simple random init.
[EM] Multistart requested: 20, prepared starts: 1
[EM] init: loglik=-6874.833794
Restricted license - for non-production use only - expires 2026-11-23
[EM] iter 001: loglik=-5600.954568  Δ=1.273879e+03  rel=1.853e-01
[EM] iter 002: loglik=-5568.681273  Δ=3.227330e+01  rel=5.762e-03
[EM] iter 003: loglik=-5545.895958  Δ=2.278531e+01  rel=4.092e-03
[EM] iter 004: loglik=-5539.146780  Δ=6.749178e+00  rel=1.217e-03
[EM] iter 005: loglik=-5516.495419  Δ=2.265136e+01  rel=4.089e-03
[EM] iter 006: loglik=-5491.900525  Δ=2.459489e+01  rel=4.458e-03
[EM] iter 007: loglik=-5446.665137  Δ=4.523539e+01  rel=8.237e-03
[EM] iter 008: loglik=-5396.858613  Δ=4.980652e+01  rel=9.144e-03
[EM] iter 009: loglik=-5380.880606  Δ=1.597801e+01  rel=2.961e-03
[EM] iter 010: loglik=-5372.473482  Δ=8.407124e+00  rel=1.562e-03
[EM] iter 011: loglik=-5367.224196  Δ=5.249286e+00  rel=9.771e-04
[EM] iter 012: loglik=-5364.075874  Δ=3

In [6]:
out['Q'] - Q_true

array([[ 0,  1,  0],
       [ 1, -1,  0],
       [ 0,  0,  0],
       [ 0,  1,  0],
       [ 1, -1,  0],
       [ 1,  1,  0],
       [ 0,  1,  0],
       [ 1, -1,  0],
       [ 0,  0,  0],
       [ 0,  0,  0],
       [ 0,  1,  0],
       [ 1, -1,  0],
       [-1,  0,  0],
       [ 0,  1,  0],
       [ 1, -1,  0],
       [ 0,  0,  0],
       [ 0,  0,  0],
       [ 0,  0,  0]])

In [ ]:
def fit(...,
        q_cfg: Optional[QMipConfig] = None,
        q_sat_cfg: Optional[QSatConfig] = None,
        ...):

    if q_cfg is None: q_cfg = QMipConfig()
    if q_sat_cfg is None:
        q_sat_cfg = QSatConfig.from_qmip(
            q_cfg,
            relax_lexi_strict=True,
            relax_distinctness=False,
            frac_assumption_sat=0.7,
            frac_sparse_maxsat=0.3,
            min_hamming_frac=0.05,
        )

In [10]:
 em.fit(R, K=K, em_cfg=em_cfg, q_cfg=q_cfg, Q_init=Q_true)

[EM] Multistart requested: 10, prepared starts: 10

[EM] === Multistart 1/10 ===
[EM] init: loglik=-5361.165105
[EM] iter 001: loglik=-5266.634800  Δ=9.453030e+01  rel=1.763e-02
[EM] iter 002: loglik=-5263.318796  Δ=3.316005e+00  rel=6.296e-04
[EM] iter 003: loglik=-5262.490398  Δ=8.283974e-01  rel=1.574e-04
[EM] iter 004: loglik=-5262.165213  Δ=3.251852e-01  rel=6.179e-05
[EM] iter 005: loglik=-5262.019134  Δ=1.460785e-01  rel=2.776e-05
[EM] iter 006: loglik=-5261.950649  Δ=6.848531e-02  rel=1.302e-05
[EM] iter 007: loglik=-5261.918028  Δ=3.262133e-02  rel=6.199e-06
[EM] iter 008: loglik=-5261.902385  Δ=1.564322e-02  rel=2.973e-06
[EM] iter 009: loglik=-5261.894858  Δ=7.526655e-03  rel=1.430e-06
[EM] iter 010: loglik=-5261.891229  Δ=3.628918e-03  rel=6.897e-07

[EM] === Multistart 2/10 ===
[EM] init: loglik=-5926.960423
[EM] iter 001: loglik=-5431.942674  Δ=4.950177e+02  rel=8.352e-02
[EM] iter 002: loglik=-5405.079041  Δ=2.686363e+01  rel=4.945e-03
[EM] iter 003: loglik=-5386.952014 

{'Q': array([[1, 0, 0],
        [0, 1, 0],
        [0, 0, 1],
        [1, 0, 0],
        [0, 1, 0],
        [0, 0, 1],
        [1, 0, 0],
        [0, 1, 0],
        [0, 0, 1],
        [1, 1, 0],
        [1, 0, 1],
        [0, 1, 1],
        [1, 1, 0],
        [1, 0, 1],
        [0, 1, 1],
        [1, 1, 1],
        [1, 1, 1],
        [1, 1, 1]]),
 's': array([0.19324413, 0.2281135 , 0.14986256, 0.23222547, 0.19161507,
        0.15931348, 0.21079798, 0.20632343, 0.16708396, 0.21684398,
        0.15686699, 0.21560071, 0.21883614, 0.2142669 , 0.17647477,
        0.14631756, 0.21135983, 0.22221749]),
 'g': array([0.20833127, 0.19907133, 0.19294261, 0.19934808, 0.18919312,
        0.26689573, 0.21825386, 0.17594493, 0.22567344, 0.20024898,
        0.20670435, 0.24433058, 0.16486987, 0.21437869, 0.18606056,
        0.23174865, 0.19906942, 0.17487708]),
 'nu': array([0.22325733, 0.0765394 , 0.12863933, 0.1165126 , 0.10886766,
        0.07645981, 0.10531514, 0.16440874]),
 'Xi': array([[0, 0, 

In [13]:
row = {
    "q_acc": q_entry_accuracy(Q_true, Q_hat),
    "q_hamming": q_hamming(Q_true, Q_hat),
    "q_row_acc": q_row_recovery(Q_true, Q_hat),
    **sg_rmse(s_true, g_true, s_hat, g_hat),
    "iters": len(out["history"]) - 1,
    "runtime_sec": dt,
}

In [14]:
row

{'q_acc': 1.0,
 'q_hamming': 0,
 'q_row_acc': 1.0,
 'rmse_s': 0.02569994542700841,
 'rmse_g': 0.019199680919762645,
 'iters': 14,
 'runtime_sec': 8.133485794067383}